<a href="https://www.kaggle.com/code/mayuringle8890/machine-learning-visualization-part-2?scriptVersionId=275823228" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# 📊 Data Visualization Tutorial - Part 02: Geographic Visualizations with Plotly

## Welcome to Part 2 of the Data Visualization Series!

In this comprehensive tutorial, we'll explore **geographic visualizations** using powerful Python libraries. This notebook is designed for beginners and covers:

### 🎯 Learning Objectives:
1. **Choropleth Maps**: Learn to create interactive world maps with color-coded regions
2. **Plotly Express**: Master the basics of this powerful visualization library
3. **Geographic Data Processing**: Work with location data, coordinates, and geocoding
4. **Interactive Visualizations**: Create engaging, clickable charts
5. **Time-Series Animations**: Visualize data changes over time

### 📚 What You'll Learn:
- How to prepare geographic data for visualization
- Creating various types of choropleth maps
- Converting location names to coordinates (geocoding)
- Customizing map colors, scales, and projections
- Adding interactivity and animations to maps
- Best practices for geographic data visualization

### 📊 Datasets Used:
- **Gapminder Dataset**: World development indicators (GDP, population, life expectancy)
- **Heart Disease UCI Dataset**: Medical data for practice visualizations

### 🛠️ Technologies:
- Plotly Express: For interactive visualizations
- Pandas: For data manipulation
- NumPy: For numerical operations
- Seaborn/Matplotlib: For additional plotting

---

## 💡 Pro Tip for Beginners:
**Take your time!** Data visualization is both an art and a science. Run each cell, observe the output, and try modifying parameters to see how visualizations change.

---

## 1️⃣ Environment Setup

First, let's import all necessary libraries. Each library serves a specific purpose:

- **pandas**: Data manipulation and analysis
- **numpy**: Numerical computing
- **seaborn**: Statistical data visualization
- **plotly.express**: Simple interface for creating interactive plots
- **matplotlib**: Low-level plotting library

We'll also configure display settings for better output formatting.

In [ ]:
# Import essential libraries for data manipulation and visualization
import pandas as pd 
import numpy as np 
import seaborn as sns 
import plotly.express as px
import matplotlib.pyplot as plt 
%matplotlib inline 

# Configure pandas to display 2 decimal places for cleaner output
pd.set_option('display.precision', 2)

# Set seaborn style with a beige background for aesthetic appeal
sns.set(rc={"axes.facecolor":"Beige", "axes.grid": False})

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings("ignore")

print("✅ All libraries imported successfully!")
print(f"📦 Pandas version: {pd.__version__}")
print(f"📦 NumPy version: {np.__version__}")

## 2️⃣ Loading and Exploring Data

### Understanding the Heart Disease Dataset

We'll start by loading the UCI Heart Disease dataset. This dataset contains medical information about patients and is commonly used for machine learning classification tasks.

**Dataset Features:**
- **age**: Patient's age in years
- **sex**: Patient's gender (1 = male, 0 = female)
- **cp**: Chest pain type (0-3)
- **trestbps**: Resting blood pressure (mm Hg)
- **chol**: Serum cholesterol (mg/dl)
- **fbs**: Fasting blood sugar > 120 mg/dl (1 = true, 0 = false)
- **restecg**: Resting electrocardiographic results (0-2)
- **thalch**: Maximum heart rate achieved
- **exang**: Exercise induced angina (1 = yes, 0 = no)
- **oldpeak**: ST depression induced by exercise
- **slope**: Slope of the peak exercise ST segment
- **ca**: Number of major vessels colored by fluoroscopy (0-3)
- **thal**: Thalassemia (0 = normal, 1 = fixed defect, 2 = reversible defect)
- **num**: Diagnosis of heart disease (target variable)

In [ ]:
# Load the Heart Disease UCI dataset from Kaggle input
df_heart = pd.read_csv("/kaggle/input/heart-disease-data/heart_disease_uci.csv")

# Remove any rows with missing values to ensure clean data
df_heart = df_heart.dropna()

# Display dataset shape (rows, columns)
print(f"📊 Dataset Shape: {df_heart.shape[0]} rows × {df_heart.shape[1]} columns")
print(f"\n✅ Dataset loaded and cleaned successfully!\n")

# Show the first few rows to understand the data structure
print("First 5 rows of the dataset:")
df_heart.head()

## 3️⃣ Geographic Data: The Gapminder Dataset

### What is Gapminder?

The **Gapminder Foundation** provides free data about global development. Their dataset includes:
- Population statistics
- GDP per capita
- Life expectancy
- Geographic regions

This data is perfect for creating **choropleth maps** (maps where areas are shaded based on data values).

### Why Use Geographic Visualizations?

Geographic visualizations help us:
1. **Identify patterns** across different regions
2. **Compare countries** or continents at a glance
3. **Track changes** over time geographically
4. **Communicate insights** effectively to diverse audiences

In [ ]:
# Load the Gapminder dataset
df = pd.read_csv('/kaggle/input/global-statistics-dataset/gapminder_full.csv')

# Remove duplicate entries to ensure data quality
df = df.drop_duplicates(ignore_index=True)

print(f"📊 Gapminder Dataset Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\n📅 Year Range: {df['Year'].min() if 'Year' in df.columns else 'N/A'} - {df['Year'].max() if 'Year' in df.columns else 'N/A'}")
print(f"\n🌍 Columns in dataset:")
print(df.columns.tolist())
print("\nFirst 5 rows:")
df.head()

## 4️⃣ Geocoding: Converting Locations to Coordinates

### What is Geocoding?

**Geocoding** is the process of converting location names (like "United States" or "Tokyo") into geographic coordinates (latitude and longitude). This is essential for plotting locations on maps.

### Why Do We Need It?

Many visualization libraries require **latitude and longitude coordinates** to plot locations accurately. If your dataset only has location names, geocoding helps you obtain these coordinates.

### The Geocoder Library

We'll use the `geocoder` library with the ArcGIS service, which provides:
- High accuracy
- Support for various location formats
- Free API access (with rate limits)

⚠️ **Important Note**: Geocoding can be slow for large datasets as it makes API calls for each location. Be patient!

In [ ]:
# Install the geocoder library for converting location names to coordinates
!pip install geocoder -q

print("✅ Geocoder library installed successfully!")

### Performing Geocoding

Now we'll add latitude and longitude coordinates to our dataset. This process:
1. Takes each region name from the 'Region' column
2. Queries the ArcGIS API for coordinates
3. Stores latitude and longitude in new columns

**Understanding the Code:**
- `lambda x: arcgis(x).latlng`: Anonymous function that geocodes each region
- `apply(func=...)`: Applies the function to each row
- `x[0]` and `x[1]`: Extracts latitude and longitude from the coordinate pair

In [ ]:
%%time
# Import required modules
import arrow
from geocoder import arcgis

# Record start time to measure geocoding duration
time_start = arrow.now()

# Check if 'Region' column exists before geocoding
if 'Region' in df.columns:
    print("🌍 Starting geocoding process...")
    print(f"Processing {df['Region'].nunique()} unique regions\n")
    
    # Apply geocoding to get latitude/longitude for each region
    df['latlng'] = df['Region'].apply(func=lambda x: arcgis(x).latlng)
    
    # Extract latitude (first element) and longitude (second element)
    df['latitude'] = df['latlng'].apply(func=lambda x: x[0] if x else None)
    df['longitude'] = df['latlng'].apply(func=lambda x: x[1] if x else None)
    
    print("\n✅ Geocoding completed!")
    print(f"\n📍 Sample coordinates:")
    print(df[['Region', 'latitude', 'longitude']].head())
else:
    print("⚠️ 'Region' column not found. Skipping geocoding.")

## 5️⃣ Creating Your First Choropleth Map

### What is a Choropleth Map?

A **choropleth map** uses different colors or shades to represent data values across geographic areas. Think of it as a "heat map" on a world map.

### Components of a Choropleth Map:

1. **locations**: Country codes or names (ISO codes preferred)
2. **locationmode**: How locations are identified (e.g., 'ISO-3' for 3-letter country codes)
3. **color**: The data column that determines shading intensity
4. **hover_name**: What appears when you hover over a region
5. **color_continuous_scale**: Color scheme (e.g., 'Viridis', 'Blues', 'Reds')
6. **projection**: Map projection style (e.g., 'natural earth', 'mercator')

### Reading the Map:

- **Darker/More Intense Colors**: Higher values
- **Lighter Colors**: Lower values
- **Hover**: Move your mouse over regions to see exact values
- **Zoom**: Use mouse wheel or touch gestures to zoom in/out

In [ ]:
# Create a choropleth map showing a specific metric across countries
# We'll visualize the first available numeric column

# Identify numeric columns for visualization
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"📊 Available numeric columns: {numeric_cols[:5]}...")  # Show first 5

# Check if we have necessary columns for a choropleth
if 'Country' in df.columns or 'country' in df.columns:
    country_col = 'Country' if 'Country' in df.columns else 'country'
    
    # Select a metric to visualize (using first numeric column as example)
    if len(numeric_cols) > 0:
        metric = numeric_cols[0]
        
        print(f"\n🗺️ Creating choropleth map for: {metric}\n")
        
        fig1 = px.choropleth(
            df,
            locations=country_col,  # Column with country names
            locationmode='country names',  # How to interpret location data
            color=metric,  # Column determining color intensity
            hover_name=country_col,  # Display country name on hover
            color_continuous_scale='Viridis',  # Color scheme (purple to yellow)
            title=f'World Map: {metric} by Country',  # Map title
            projection='natural earth',  # Map projection style
            height=600,  # Height in pixels
            labels={metric: metric.replace('_', ' ').title()}  # Clean label
        )
        
        # Customize layout for better appearance
        fig1.update_layout(
            geo=dict(
                showframe=False,  # Remove frame
                showcoastlines=True,  # Show coastlines
                projection_type='natural earth'  # Projection type
            )
        )
        
        fig1.show()
        
        print("\n💡 Tip: Hover over countries to see exact values!")
    else:
        print("⚠️ No numeric columns found for visualization")
else:
    print("⚠️ No country column found. Using alternative approach...")
    # Try using Plotly's built-in gapminder dataset as fallback
    df_gapminder = px.data.gapminder()
    print(f"\n📊 Using built-in Gapminder data: {df_gapminder.shape}")
    print(f"Columns: {df_gapminder.columns.tolist()}")

### 🎨 Understanding Color Scales

Color scales are crucial for effective data visualization. Here are popular options:

**Sequential Scales** (for continuous data):
- `'Viridis'`: Purple → Green → Yellow (colorblind-friendly)
- `'Blues'`: Light Blue → Dark Blue
- `'Reds'`: Light Red → Dark Red
- `'Greens'`: Light Green → Dark Green

**Diverging Scales** (for data with a meaningful midpoint):
- `'RdBu'`: Red → White → Blue
- `'RdYlGn'`: Red → Yellow → Green
- `'Spectral'`: Red → Orange → Yellow → Green → Blue

**When to Use Each:**
- Use **sequential** for data that goes from low to high (e.g., population, GDP)
- Use **diverging** for data with a meaningful center (e.g., temperature change, profit/loss)

💡 **Best Practice**: Always choose colorblind-friendly palettes when possible!

## 6️⃣ Creating Multiple Choropleth Variations

Let's explore different ways to customize choropleth maps. We'll create variations with:
1. Different color scales
2. Different projections
3. Different hover information

### Map Projections Explained:

- **'natural earth'**: Balanced, aesthetically pleasing (default)
- **'mercator'**: Traditional web maps projection (distorts size near poles)
- **'orthographic'**: Globe-like 3D appearance
- **'equirectangular'**: Simple rectangular projection

In [ ]:
# Let's use Plotly's built-in gapminder dataset for comprehensive examples
df_gapminder = px.data.gapminder()

print("📊 Built-in Gapminder Dataset Loaded")
print(f"Shape: {df_gapminder.shape}")
print(f"\nColumns: {df_gapminder.columns.tolist()}")
print(f"\nYears available: {df_gapminder['year'].min()} - {df_gapminder['year'].max()}")
print(f"Countries: {df_gapminder['country'].nunique()}")

# Show sample data
print("\nSample data:")
df_gapminder.head()

### Variation 1: Life Expectancy Across the World

This map shows how long people are expected to live in different countries. 

**What to Look For:**
- Developed countries typically have higher life expectancy (darker colors)
- Regional patterns (e.g., Europe vs. Africa)
- Changes over time (we'll filter for a specific year)

In [ ]:
# Filter for most recent year to see current state
latest_year = df_gapminder['year'].max()
df_latest = df_gapminder[df_gapminder['year'] == latest_year]

print(f"📅 Visualizing data for year: {latest_year}\n")

# Create choropleth for life expectancy
fig2 = px.choropleth(
    df_latest,
    locations='iso_alpha',  # 3-letter country codes (ISO standard)
    locationmode='ISO-3',  # Use ISO-3 country codes
    color='lifeExp',  # Life expectancy column
    hover_name='country',  # Show country name on hover
    hover_data={'pop': ':,', 'gdpPercap': ':,.0f', 'lifeExp': ':.1f'},  # Additional info
    color_continuous_scale='RdYlGn',  # Red (low) to Green (high)
    title=f'Life Expectancy by Country ({latest_year})',
    projection='natural earth',
    labels={'lifeExp': 'Life Expectancy (years)', 
            'pop': 'Population',
            'gdpPercap': 'GDP per Capita'}
)

# Enhance appearance
fig2.update_layout(
    height=600,
    geo=dict(
        showframe=False,
        showcoastlines=True,
        coastlinecolor='Gray'
    ),
    font=dict(size=12)
)

fig2.show()

print("\n📊 Interpretation Guide:")
print("🟢 Green countries: Higher life expectancy (75+ years)")
print("🟡 Yellow countries: Medium life expectancy (60-75 years)")
print("🔴 Red countries: Lower life expectancy (<60 years)")

### Variation 2: GDP Per Capita Distribution

**GDP per capita** measures the average economic output per person. It's a key indicator of a country's wealth and development level.

**Understanding GDP per Capita:**
- Higher values = More prosperous economy
- Often correlates with life expectancy and quality of life
- Measured in international dollars (purchasing power parity)

**What to Observe:**
- Wealthy nations cluster in specific regions
- Resource-rich countries may have high GDP
- Historical and political factors influence economic development

In [ ]:
# Create choropleth for GDP per capita
fig3 = px.choropleth(
    df_latest,
    locations='iso_alpha',
    locationmode='ISO-3',
    color='gdpPercap',  # GDP per capita column
    hover_name='country',
    hover_data={
        'pop': ':,',  # Format population with commas
        'lifeExp': ':.1f',  # Show 1 decimal place
        'gdpPercap': ':$,.0f'  # Format as currency
    },
    color_continuous_scale='Plasma',  # Purple to Yellow scale
    title=f'GDP per Capita by Country ({latest_year})',
    projection='mercator',  # Different projection for variety
    labels={
        'gdpPercap': 'GDP per Capita (USD)',
        'pop': 'Population',
        'lifeExp': 'Life Expectancy'
    },
    range_color=[0, 50000]  # Set color range for better contrast
)

fig3.update_layout(
    height=600,
    geo=dict(
        showframe=False,
        showcoastlines=True,
        projection_type='mercator'
    )
)

fig3.show()

print("\n💰 Economic Insights:")
print("- Lighter colors indicate higher GDP per capita")
print("- Notice the correlation with developed vs. developing nations")
print("- Some small countries may have very high GDP per capita")

## 7️⃣ Animated Choropleth Maps: Time-Series Visualization

### Why Animations?

Animated maps let you see **how data changes over time**. Instead of creating separate maps for each year, animations show the progression in a single, engaging visualization.

**Benefits of Animated Maps:**
1. **Identify Trends**: See which countries improve or decline over time
2. **Tell Stories**: Narrative power to show historical changes
3. **Compare Rates**: Observe which regions change faster
4. **Engage Viewers**: Interactive and memorable

### How to Read an Animated Map:

1. **Play Button**: Click to start the animation
2. **Pause Button**: Stop at any point to examine details
3. **Slider**: Drag to jump to specific years
4. **Speed**: Most animations show 1-2 years per second

### Creating an Animated Choropleth

We'll visualize how life expectancy has changed globally from 1952 to 2007.

In [ ]:
# Create an animated choropleth showing life expectancy over time
print("🎬 Creating animated visualization...\n")

fig_animated = px.choropleth(
    df_gapminder,  # Use full dataset (all years)
    locations='iso_alpha',
    locationmode='ISO-3',
    color='lifeExp',
    hover_name='country',
    hover_data={'pop': ':,', 'gdpPercap': ':$,.0f', 'lifeExp': ':.1f'},
    animation_frame='year',  # THIS CREATES THE ANIMATION!
    animation_group='country',  # Group by country for smooth transitions
    color_continuous_scale='Viridis',
    range_color=[20, 90],  # Fixed range for consistent comparison across years
    title='Life Expectancy Evolution (1952-2007)',
    projection='natural earth',
    labels={
        'lifeExp': 'Life Expectancy (years)',
        'pop': 'Population',
        'gdpPercap': 'GDP per Capita'
    },
    height=650
)

# Customize layout
fig_animated.update_layout(
    geo=dict(
        showframe=False,
        showcoastlines=True,
        projection_type='natural earth'
    ),
    font=dict(size=13)
)

# Slow down animation for better viewing
fig_animated.layout.updatemenus[0].buttons[0].args[1]['frame']['duration'] = 1000  # 1 second per frame
fig_animated.layout.updatemenus[0].buttons[0].args[1]['transition']['duration'] = 500  # Smooth transition

fig_animated.show()

print("\n🎯 What to Observe:")
print("1. Overall global improvement in life expectancy")
print("2. Developed nations start with higher values and improve steadily")
print("3. Developing nations show rapid improvement in recent decades")
print("4. Some regions lag behind due to various factors (wars, diseases, poverty)")
print("\n💡 Try pausing the animation to examine specific years!")

### Historical Context

The animation reveals several important historical trends:

**1950s-1960s:**
- Post-WWII recovery in Europe and Asia
- Medical advances (antibiotics, vaccines)
- Large gaps between developed and developing nations

**1970s-1980s:**
- Green Revolution improves food security
- Continued medical progress
- Some countries experience setbacks (conflicts, economic crises)

**1990s-2000s:**
- Globalization spreads medical knowledge
- HIV/AIDS impacts some regions
- Emerging economies show rapid improvement
- Gap narrows between developed and developing nations

## 8️⃣ Population Visualization

### Understanding Population Data

Population is a fundamental demographic measure. Visualizing it helps us:
- Identify the most populous nations
- Understand population distribution
- Predict resource needs and challenges
- Plan for urbanization and development

### Logarithmic Scales

Population varies enormously between countries (from thousands to billions). We often use **logarithmic scales** to visualize such wide ranges.

**What is a Log Scale?**
- Regular scale: 0, 10, 20, 30, 40...
- Log scale: 1, 10, 100, 1000, 10000...
- Each step is a multiplication, not addition
- Helps show both small and large values clearly

In [ ]:
# Create animated population visualization
print("🌍 Creating population animation...\n")

fig_pop = px.choropleth(
    df_gapminder,
    locations='iso_alpha',
    locationmode='ISO-3',
    color='pop',  # Population column
    hover_name='country',
    hover_data={
        'pop': ':,',  # Format with commas
        'lifeExp': ':.1f',
        'gdpPercap': ':$,.0f'
    },
    animation_frame='year',
    animation_group='country',
    color_continuous_scale='YlOrRd',  # Yellow to Red scale
    title='World Population by Country (1952-2007)',
    projection='natural earth',
    labels={
        'pop': 'Population',
        'lifeExp': 'Life Expectancy',
        'gdpPercap': 'GDP per Capita'
    },
    range_color=[0, 1_000_000_000],  # Fixed range up to 1 billion
    height=650
)

fig_pop.update_layout(
    geo=dict(
        showframe=False,
        showcoastlines=True
    )
)

# Adjust animation speed
fig_pop.layout.updatemenus[0].buttons[0].args[1]['frame']['duration'] = 1000
fig_pop.layout.updatemenus[0].buttons[0].args[1]['transition']['duration'] = 500

fig_pop.show()

print("\n📊 Population Insights:")
print("- China and India dominate in population (darkest red)")
print("- Population growth accelerates in Africa and Asia")
print("- Europe shows slower population growth")
print("- Small countries may not be visible due to scale")

## 9️⃣ Best Practices for Geographic Visualizations

### ✅ Do's:

1. **Choose Appropriate Color Scales**
   - Sequential for ordered data (population, GDP)
   - Diverging for data with meaningful midpoints (temperature change)
   - Colorblind-friendly palettes (Viridis, Cividis)

2. **Add Context with Hover Information**
   - Include country names
   - Show multiple related metrics
   - Format numbers properly (commas, currency symbols)

3. **Use Appropriate Projections**
   - Natural Earth: General purpose, visually balanced
   - Mercator: Familiar to web users
   - Orthographic: Emphasizes global nature

4. **Set Consistent Scales for Comparisons**
   - Fixed color ranges for time-series animations
   - Same projection across related maps
   - Comparable legends

5. **Provide Clear Titles and Labels**
   - What is being shown?
   - What year or time period?
   - What are the units?

### ❌ Don'ts:

1. **Don't Use Rainbow Color Scales**
   - Not colorblind-friendly
   - Can create false perceptual boundaries
   - Use perceptually uniform scales instead

2. **Don't Forget Missing Data**
   - Clearly indicate regions with no data
   - Explain why data might be missing

3. **Don't Overcrowd Maps**
   - Too much information = confusion
   - Focus on one main message per map

4. **Don't Ignore Map Distortions**
   - All flat maps distort the Earth's surface
   - Choose projections that minimize distortion for your use case

5. **Don't Use Geographic Visualizations for Everything**
   - Use maps when geography matters
   - Sometimes bar charts or scatter plots are clearer

## 🔟 Advanced Example: Multi-Metric Comparison

Let's create a more sophisticated visualization that combines multiple metrics. We'll look at the relationship between GDP and life expectancy using bubble size for population.

In [ ]:
# Create a scatter geo plot (bubbles on map)
fig_scatter = px.scatter_geo(
    df_latest,
    locations='iso_alpha',
    locationmode='ISO-3',
    color='lifeExp',  # Color represents life expectancy
    size='pop',  # Bubble size represents population
    hover_name='country',
    hover_data={
        'pop': ':,',
        'gdpPercap': ':$,.0f',
        'lifeExp': ':.1f'
    },
    color_continuous_scale='Turbo',
    title=f'Life Expectancy, GDP, and Population ({latest_year})',
    projection='natural earth',
    size_max=50,  # Maximum bubble size
    labels={
        'lifeExp': 'Life Expectancy (years)',
        'pop': 'Population',
        'gdpPercap': 'GDP per Capita'
    },
    height=650
)

fig_scatter.update_layout(
    geo=dict(
        showframe=False,
        showcoastlines=True,
        showcountries=True,
        countrycolor='lightgray'
    )
)

fig_scatter.show()

print("\n📊 Reading This Visualization:")
print("🎨 Color: Life Expectancy (Blue = low, Red = high)")
print("⭕ Bubble Size: Population (Larger = more people)")
print("📍 Location: Geographic position")
print("\n💡 This shows THREE variables simultaneously!")

## 📚 Summary and Key Takeaways

Congratulations! You've completed Part 2 of the Data Visualization Tutorial. Let's recap what you've learned:

### 🎓 Skills Acquired:

1. **Choropleth Maps**
   - Created interactive world maps
   - Used different color scales and projections
   - Added hover information for context

2. **Geocoding**
   - Converted location names to coordinates
   - Understood the importance of geographic data

3. **Animated Visualizations**
   - Created time-series animations
   - Showed data evolution over decades
   - Customized animation speed and transitions

4. **Multi-Dimensional Visualizations**
   - Combined multiple metrics (color, size, location)
   - Created scatter geo plots

5. **Best Practices**
   - Chose appropriate color scales
   - Provided clear labels and context
   - Made visualizations accessible

### 🚀 Next Steps:

Ready to continue your data visualization journey? Check out:

- **Part 03**: Joint plots and statistical visualizations with Seaborn
- **Part 04**: 3D visualizations and advanced plotting techniques
- **Part 05**: Missing data visualization and advanced preprocessing

### 💪 Practice Exercises:

Try these to reinforce your learning:

1. Create a choropleth showing GDP per capita with a different color scale
2. Make an animated map showing population growth
3. Experiment with different map projections
4. Add your own dataset with geographic data
5. Create a dashboard with multiple related maps

### 📖 Additional Resources:

- [Plotly Documentation](https://plotly.com/python/)
- [Gapminder Foundation](https://www.gapminder.org/)
- [ColorBrewer](https://colorbrewer2.org/) - Choose better colors
- [Datawrapper Academy](https://academy.datawrapper.de/) - Visualization best practices

---

### 🙏 Thank You!

Thank you for completing this tutorial. Remember:
- **Practice makes perfect** - keep creating visualizations!
- **Experiment** - try different settings and approaches
- **Share** - show your work and get feedback
- **Learn continuously** - data visualization is an evolving field

Happy visualizing! 📊✨

---

## 📝 Notebook Information

**Title**: Data Visualization Tutorial - Part 02: Geographic Visualizations  
**Author**: Enhanced for Kaggle  
**Purpose**: Educational tutorial for beginners in data visualization  
**Topics**: Choropleth maps, Plotly, Geographic data, Animations  
**Difficulty**: Beginner to Intermediate  
**Estimated Time**: 60-90 minutes  

**Datasets Required**:
- Gapminder (global development indicators)
- Heart Disease UCI (optional, for additional practice)

**Libraries Used**:
```
pandas>=1.3.0
numpy>=1.21.0
plotly>=5.0.0
seaborn>=0.11.0
matplotlib>=3.4.0
geocoder>=1.38.1
```

---